In [2]:
"""
This file implements BitFit, which is a parameter efficient fine-tuning method.
It only updates the biases and not the weights.
"""


'\nThis file implements BitFit, which is a parameter efficient fine-tuning method.\nIt only updates the biases and not the weights.\n'

In [3]:
# from scripts.task3_data_preparation import prepare_data_qu
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets
import sys
import os
import evaluate
import numpy as np
import torch
from tqdm import tqdm

# To use the primary GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


# Get the current working directory (which should be inside the 'notebooks' directory)
current_dir = os.getcwd()

# Go up one level to the parent directory
parent_dir = os.path.dirname(current_dir)

# Add the 'scripts' directory to the Python path
scripts_path = os.path.join(parent_dir, 'scripts')
sys.path.insert(0, scripts_path)

In [4]:
def compute_metrics(eval_pred, metric):
    """
    This function will be used for model evaluation. 

    args:
        eval_pred: the output of the model. Made up of logits and labels, where logits
        are unnormalized scores that the model predicts. These logits are typically 
        transformed into probabilities using a softmax function. 

    returns:
        - Dictionary: the keys represent different sub-datasets and the evaluation on them separately.
    """

    # In evaluation mode, the output of an autoregressive model like XGLM consists of logits 
    # corresponding to the score for each possible next token in the vocabulary given the previous sequence of tokens. 
    # The label is the ground truth—the actual token that comes next in the sequence according to the dataset, 
    # regardless of the model's prediction. It's the target that the model is trying to predict.
    logits, labels = eval_pred

    # Convert logits to probabilities for all possible next words
    probabilities = torch.softmax(logits, axis=-1)
    
    # You need the probabilities of the actual next tokens, referenced by the labels
    true_next_token_probabilities = np.take_along_axis(probabilities, np.expand_dims(labels, -1), axis=-1)

    # Perplexity is calculated using the probabilities of the true next tokens
    perplexity = metric.compute(predictions=true_next_token_probabilities.squeeze(), references=labels)
    
    return perplexity

In [5]:
# Steps
# 1) Prepare training data
# 2) pass in a batch of data to the model
# 3) compute the loss
# 4) update the bias parameters based on gradient 

In [6]:
# Load the dataset
dataset_path = "hackathon-pln-es/spanish-to-quechua"

data_files = {
"train": "data/train-00000-of-00001.parquet",
"validation": "data/validation-00000-of-00001.parquet",
"test": "data/test-00000-of-00001.parquet"
}
# load the dataset for training
dataset = datasets.load_dataset(dataset_path, data_files=data_files)
training_data = dataset["train"]

# prepare the dataloader
training_data_loader =  DataLoader(training_data, batch_size=64, shuffle=True)


In [7]:
# define the model
MODEL_NAME = "facebook/xglm-564M"

# load pre-trained model from the huggingface hub
xglm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# set to train mode
xglm_model.train()

# load a pre-trained tokenizer from the huggingface hub
xglm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# since we are using a decoder only architecture, we need to perform the padding
# on the left side so that the model does not get confused while guessing the next token.
xglm_tokenizer.padding_side = 'left'


In [8]:
# freeze the weights and allow only the biases to get updated.
# BitFit updates only the bias terms and not the weights. 
# The bias terms are the additive constants in various layers of 
# the model, hence, fine tuning these parameters alone is very efficient.
 
# iterate over the model parameters
for name, param in xglm_model.named_parameters():
    # if the parameter is a weight, no need to calculate the gradient.
    if 'weight' in name:
        param.requires_grad = False
    elif 'bias' in name:
        param.requires_grad = True


In [9]:
# define the optimizer
# we need to tell the optimizer to only update the 
# parameters which require gradient calculation (biases for BitFit)
optimizer = torch.optim.SGD(filter(lambda p: p.requires_grad, xglm_model.parameters()), lr=0.001, momentum=0.9)

# Setup evaluation, which will be used to calculate the loss.
metric = evaluate.load("perplexity", module_type="metric")

# define the loss function
loss_fn = torch.nn.CrossEntropyLoss()

# remember to freeze the weights and not the biases

In [10]:
# start the fine-tuning process
for i, data in enumerate(tqdm(training_data_loader, desc="Training")):
    
    # Zero your gradients for every batch!
    optimizer.zero_grad()
    
    # only get the "qu" sentences
    qu_sentences = data["qu"]
   
    # tokenize the sentences tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    tokenized = xglm_tokenizer(qu_sentences, padding=True, truncation=True, return_tensors="pt")
    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    # define labels: 
    labels=input_ids.clone()
    
    # Make predictions for this batch
    outputs = xglm_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    # the shape of logits is e.g. [6 x 32 x 256008] where 256008 is the vocab size,
    # 32 is the number of token in each sentence, and 6 is the size of the batch.
    logits = outputs["logits"]
    
    # labels => torch.Size([batch_size, # of token per sentence]). logits => torch.Size([batch_size, # of token per sentence, vocab_size])
    # print(labels.shape, logits.shape) 

    logits_flat = logits.view(-1, logits.size(-1))
    labels_flat = labels.view(-1)

    # Compute the loss
    loss = loss_fn(logits_flat, labels_flat)
   
    # Backward pass
    loss.backward()

    # Update the parameters
    optimizer.step()

Training:   0%|          | 0/1606 [00:00<?, ?it/s]

64
test1
test2
test3


Training:   0%|          | 1/1606 [00:14<6:15:37, 14.04s/it]

test4
test5
64
test1
test2
test3


Training:   0%|          | 2/1606 [00:24<5:24:10, 12.13s/it]

...

test4
test5
64
test1
test2
test3


Training:   2%|▏         | 29/1606 [06:21<5:39:20, 12.91s/it]

test4
test5
64
test1
test2
test3


Training:   2%|▏         | 30/1606 [06:32<5:25:48, 12.40s/it]

test4
test5
64
test1
test2
test3
